# AKANE E10：辞書の確かめは、別の問い（部分と全体）・別の分野（病気・地理）でも効くか

このノートは **材料を取るだけ** です（比べは結果のファイルを読んで別に行います）。計画：sims/e10/PLAN_E10.md。

- 3つの場面の主張 約4.7万（部分と全体・病気・地理、物×分類の全部の組）。正解はこのノートに入っていません。
- Gemmaに全部の主張を「はい／いいえ」で聞きます（部分と全体は逆向きの文も）。
- Wiktionaryのダンプを取り、名詞と固有名詞（地名）の説明文を読みます。
- E8と同じ鎖（説明文をたどる、途中の語の意味はGemmaが選ぶ）に、「〜の一部」「〜にある」の段を足し、NLIで判定します。

## 手順
1. 「ランタイム」→「ランタイムのタイプを変更」→ **L4 GPU**。
2. 鍵マークの `HF_TOKEN` にアクセスを許可（E8と同じ）。
3. 「ランタイム」→「すべてのセルを実行」。ドライブの許可を聞かれたら許可。
4. 終わると、ドライブの `AKANE_アーキテクチャ/E10_results/` に `E10_*.json.gz`（数個）ができます。最後のセルが一覧と大きさを出します。
5. 時間の見込み：Gemmaの「はい／いいえ」10〜15分、ダンプを取って読む 30〜50分（この間「GPUを使っていません」の警告が出ますが×で閉じてください）、語の一覧 5〜10分、Gemmaの意味選び 15〜25分、NLI 90〜150分。合わせて3〜4時間。
6. 途中で切れても、もう一度「すべてのセルを実行」すれば、済んだ所は飛ばします（途中の控えをドライブに保存しています）。

In [ ]:
import os, json, time, bz2, gzip, subprocess, sys, hashlib
import numpy as np
from google.colab import drive, userdata
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/AKANE_アーキテクチャ"
OUT = f"{BASE}/E10_results"
os.makedirs(OUT, exist_ok=True)
NLI_ID = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"   # E6〜E9と同じ
GEMMA_ID = "google/gemma-2-2b-it"
DUMP_URL = "https://dumps.wikimedia.org/enwiktionary/latest/enwiktionary-latest-pages-articles.xml.bz2"
subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"], check=False)
def save_gz(obj, name):
    p = f"{OUT}/{name}.json.gz"
    with gzip.open(p, "wt", encoding="utf-8") as f: json.dump(obj, f, ensure_ascii=False)
    print("保存:", name, round(os.path.getsize(p) / 1e6, 2), "MB"); return p
def load_gz(name):
    p = f"{OUT}/{name}.json.gz"
    if not os.path.exists(p): return None
    with gzip.open(p, "rt", encoding="utf-8") as f: return json.load(f)

In [ ]:
# 中身（sims/e10/e10_core.py と同じ）
"""E10の中身（Colabと手元の両方で使う）。E8（sims/e8/e8_core.py）からの変更点：
1. ダンプは「名詞」に加えて「固有名詞」（地名）も読み、空白入り（4語まで）・大文字始まりの見出しも取る。
   地名の説明文によく使われる {{place|en|city|p/Osaka|c/Japan}} を「A city in Osaka, Japan.」の形の文に直す。
2. 鎖の段を2種類にする：
   - 一種の段（E8と同じ、説明文の頭の名詞）
   - 一部・場所の段（新）：説明文の中の「N of Y」「N in Y」で、N が場面ごとの決まった語の一覧に入る時だけ Y をたどる（狭いたどり方）。
     広いたどり方（N を問わない）は部分と全体の参考だけ。
3. 段の確からしさのNLIの言い方：一種の段は E8 と同じ「w is a type of g.」、一部・場所の段は場面ごとに
   P「w is a part of g.」 G「w is located in g.」 D「w affects the g.」
4. 着き（点→分類）のNLIの言い方：P「w is a part of (全体).」 G「w is located in (地域).」
   D「w is a (分類).」と、体の部位の分類なら「w is a part of the (器官系).」の大きい方。
意味の選び方（Gemmaに番号で選ばせる D・印のない最初の意味 B）、3段まで、一番強い道の掛け算は E8 と同じ。
"""
import re

MAX_DEFS_ENT, MAX_DEFS_MID, DEPTH, MAX_LOC = 8, 8, 3, 3
BAD_LABELS = {"obsolete", "archaic", "dated", "rare", "slang", "vulgar", "offensive", "derogatory", "historical",
              "humorous", "euphemistic", "nonstandard", "pejorative", "ethnic slur", "slur", "obsolete outside dialects",
              "rare outside dialects", "nonce word", "jocular", "informal euphemism"}
PERSON_HEADS = ("one who", "someone who", "somebody who", "a person who", "person who", "one that", "anyone who",
                "a person", "someone", "somebody")
OF_HEADS = {"kind", "type", "sort", "variety", "species", "form", "member", "one", "any", "piece", "item", "genus",
            "class", "breed", "style", "group", "family", "unit", "strain", "subspecies", "category", "instance",
            "example", "specimen", "individual", "portion", "quantity"}
META = {"synonym", "ellipsis", "term", "name", "word", "abbreviation", "plural", "spelling", "who", "something",
        "anything", "that", "which", "this", "one", "any", "form", "use", "sense", "variant", "misspelling", "acronym"}

# 一部・場所の段をたどってよい N（結果を見る前に固定、PLAN_E10.md）
LOC_HEADS = {
    "P": {"part", "piece", "section", "portion", "end", "edge", "side", "front", "back", "top", "bottom", "surface",
          "component", "organ", "segment", "bone", "muscle", "joint", "layer", "membrane", "tissue", "cavity", "area",
          "region", "structure", "tip", "rim", "frame", "cover", "covering", "opening", "lining", "wall", "base"},
    "G": {"city", "town", "capital", "port", "village", "municipality", "country", "nation", "state", "province",
          "region", "island", "archipelago", "republic", "kingdom", "territory", "prefecture", "county", "district",
          "part", "area", "emirate", "commune", "settlement", "metropolis", "suburb", "borough", "seaport", "community",
          "governorate", "department", "division", "subdivision", "regency", "oblast", "canton", "parish", "locality"},
    "D": {"disease", "disorder", "inflammation", "infection", "cancer", "tumor", "tumour", "condition", "carcinoma",
          "syndrome", "illness", "cyst", "lesion", "swelling", "injury", "ailment", "malady", "failure", "deficiency",
          "abnormality", "dysfunction", "pain", "enlargement", "narrowing", "hardening", "ulcer", "sarcoma", "lymphoma",
          "leukemia", "leukaemia", "neoplasm", "malformation", "defect", "obstruction", "infestation"}}
LOC_PREPS = {"of", "in", "within", "inside"}


def _first_head(doc):
    chunks = list(doc.noun_chunks)
    return chunks[0].root if chunks else None


def genus(defn, nlp, doc=None):
    t = re.sub(r"\([^)]*\)", " ", defn).strip()
    low = t.lower().lstrip("“\"' ")
    for p in PERSON_HEADS:
        if low.startswith(p + " ") or low == p:
            return "person"
    doc = doc if doc is not None else nlp(t[:200])
    root = _first_head(doc)
    if root is None:
        return None
    lem = root.lemma_.lower()
    if lem in OF_HEADS:
        for ch in root.children:
            if ch.dep_ == "prep" and ch.text.lower() == "of":
                objs = [c for c in ch.children if c.dep_ == "pobj"]
                if objs:
                    lem = objs[0].lemma_.lower()
                    break
    if lem == "being" and "human" in list(doc.noun_chunks)[0].text.lower():
        lem = "human"
    if lem == "specie":
        lem = "species"
    if lem in META or not re.fullmatch(r"[a-z][a-z\-]*", lem):
        return None
    return lem


def _np_text(tok):
    """前置詞の目的語のまとまり。固有名詞なら複合語ごと（"East Asia"）、ふつうの名詞は見出しの形（"lungs"→"lung"）。"""
    if tok.pos_ == "PROPN":
        left = [c for c in tok.lefts if c.dep_ in ("compound", "amod") and c.pos_ in ("PROPN", "ADJ") and c.text[:1].isupper()]
        i0 = min([c.i for c in left] + [tok.i])
        return " ".join(t.text for t in tok.doc[i0:tok.i + 1])
    lem = tok.lemma_.lower()
    return lem if re.fullmatch(r"[a-z][a-z\-]*", lem) and lem not in META else None


def loc_targets(defn, nlp, dom, wide=False, doc=None):
    """「N of Y」「N in Y」の Y（N が場面の一覧にある時、wide なら N を問わない）。P では「part of Y」は N を問わない。"""
    t = re.sub(r"\([^)]*\)", " ", defn).strip()
    doc = doc if doc is not None else nlp(t[:200])
    heads = LOC_HEADS[dom]
    out = []
    for tok in doc:
        if tok.dep_ != "prep" or tok.text.lower() not in LOC_PREPS:
            continue
        n = tok.head
        if n.pos_ not in ("NOUN", "PROPN"):
            continue
        if not (wide or n.lemma_.lower() in heads):
            continue
        for o in [c for c in tok.children if c.dep_ == "pobj"]:
            objs = [o] + [c for c in o.children if c.dep_ in ("conj", "appos") and c.pos_ == o.pos_]
            for y in objs:
                s = _np_text(y)
                if s and s not in out:
                    out.append(s)
    return out[:MAX_LOC]


# --- ダンプの読み取り（E8と同じ書き換え＋{{place}}） ---
DROP_T = {"lb", "lbl", "label", "q", "qual", "qualifier", "i", "gloss", "gl", "sense", "s", "defdate", "rfdef", "rfclarify",
          "senseid", "tlb", "def-date", "c", "C", "topics", "cat", "rfex", "rfquote", "nb...", "ux", "uxi", "quote-book"}
ARG2_T = {"l", "m", "link", "mention", "ll", "l-self", "m-self", "cog"}
ARG1_T = {"w", "taxlink", "taxfmt", "vern", "ws", "pedia", "non-gloss", "n-g", "ngd", "non-gloss definition", "glossary",
          "smallcaps", "taxon", "spelink", "nowrap", "small", "sub", "sup"}
SKIP_LINE = ("plural of", "alternative form of", "alternative spelling of", "obsolete form of", "misspelling of",
             "abbreviation of", "initialism of", "acronym of", "obsolete spelling of", "archaic spelling of", "eye dialect of",
             "nonstandard spelling of", "synonym of", "clipping of", "short for", "ellipsis of", "contraction of")
SKIP_T = re.compile(r"^\{\{\s*(plural of|alternative form of|alt form|alt sp|alternative spelling of|obsolete form of|"
                    r"misspelling of|abbreviation of|abbr of|initialism of|acronym of|obsolete spelling of|archaic spelling of|"
                    r"eye dialect of|nonstandard spelling of|synonym of|syn of|clipping of|short for|ellipsis of|"
                    r"contraction of|form of|inflection of|en-past of)\s*\|", re.I)
LB = re.compile(r"\{\{\s*(?:lb|lbl|label|tlb)\s*\|\s*en\s*\|([^{}]*)\}\}")


def _holo(h):
    h = h.split("/")[-1].strip()
    return re.sub(r"^the ", "", h)


def render_place(args):
    """{{place|en|...}} を文に。新しい書き方（<<city>> in <<p/Osaka>>）と古い書き方（city|p/Osaka|c/Japan）の両方。"""
    pos = [a for a in args if "=" not in a]
    if pos and pos[0] == "en":
        pos = pos[1:]
    if not pos:
        return ""
    if any("<<" in a for a in pos):
        txt = "|".join(pos)
        txt = re.sub(r"<<([^<>]*)>>", lambda m: _holo(m.group(1)) if "/" in m.group(1) else m.group(1), txt)
        return txt.replace("|", " ")
    groups, cur = [], []
    for a in pos:
        if a.strip() == ";":
            if cur: groups.append(cur)
            cur = []
        else:
            cur.append(a.strip())
    if cur: groups.append(cur)
    g = groups[0]
    ptype = g[0].replace("/", " ").strip()
    holos = [_holo(h) for h in g[1:] if h and h not in (",", "in", "of")]
    s = ("an " if ptype[:1].lower() in "aeiou" else "a ") + ptype
    return s + (" in " + ", ".join(holos) if holos else "")


def _tmpl(m):
    parts = [p.strip() for p in m.group(1).split("|")]
    name, args = parts[0].strip(), parts[1:]
    pos = [p for p in args if "=" not in p]
    if name == "place":
        return render_place(args)
    if name in DROP_T:
        return ""
    if name in ARG2_T:
        return pos[1] if len(pos) > 1 else ""
    if name in ARG1_T:
        return pos[0] if pos else ""
    return ""


def strip_wikitext(t):
    for _ in range(6):
        t2 = re.sub(r"\{\{([^{}]*)\}\}", _tmpl, t)
        if t2 == t:
            break
        t = t2
    t = re.sub(r"\[\[(?:[^\]|]*\|)?([^\]]*)\]\]", r"\1", t)
    t = re.sub(r"'{2,}|<[^>]+>", "", t)
    t = re.sub(r"\s+", " ", t).strip(" ;,.")
    return (t[:1].upper() + t[1:] + ".") if t else ""


def labels_of(raw):
    out = []
    for m in LB.finditer(raw):
        for p in m.group(1).split("|"):
            p = p.strip().lower()
            if p and "=" not in p and p not in ("_", "and", "or"):
                out.append(p)
    return out


def noun_senses_from_wikitext(wt, limit=MAX_DEFS_MID):
    """名詞と固有名詞の意味 [[説明文, [印]], ...]（ページに出る順）"""
    m = re.search(r"(?ms)^==English==\s*$(.*?)(?=^==[^=]|\Z)", wt or "")
    if not m:
        return []
    out = []
    for sec in re.finditer(r"(?ms)^===+\s*(?:Noun|Proper noun)\s*===+\s*$(.*?)(?=^===|\Z)", m.group(1)):
        for line in sec.group(1).splitlines():
            if not line.startswith("# "):
                continue
            raw = line[2:].strip()
            if SKIP_T.match(raw):
                continue
            d = strip_wikitext(raw)
            if len(d) > 3 and not d.lower().startswith(SKIP_LINE):
                out.append([d[:220], labels_of(raw)])
    return out[:limit]


TITLE_OK = re.compile(r"[A-Za-z][A-Za-z\-'.]*(?: [A-Za-z][A-Za-z\-'.]*){0,3}")


def parse_dump(fobj, log=print, every=500000):
    import xml.etree.ElementTree as ET
    out, n, title, ns = {}, 0, None, None
    for ev, el in ET.iterparse(fobj, events=("end",)):
        tag = el.tag.rsplit("}", 1)[-1]
        if tag == "title":
            title = el.text
        elif tag == "ns":
            ns = el.text
        elif tag == "text":
            if ns == "0" and title and TITLE_OK.fullmatch(title) and el.text and "==English==" in el.text:
                ds = noun_senses_from_wikitext(el.text)
                if ds:
                    out[title] = ds
        elif tag == "page":
            n += 1
            el.clear()
            if n % every == 0:
                log(f"ページ {n:,}  名詞のある語 {len(out):,}")
    return out


def is_bad(labels):
    return any(l in BAD_LABELS for l in labels)


def first_good(senses):
    for k, s in enumerate(senses):
        if not is_bad(s["labels"] if isinstance(s, dict) else s[1]):
            return k
    return 0


def lookup(WKD, w):
    """見出しを探す：そのまま → 先頭だけ大文字/小文字を変えて → 'the ' を外して。"""
    for v in (w, w[:1].upper() + w[1:], w[:1].lower() + w[1:], re.sub(r"^[Tt]he ", "", w)):
        if v in WKD:
            return v
    return None


# --- 鎖 ---
def build_lexicon(ents_by_dom, WKD, nlp, log=print, wide_doms=("P",)):
    """L[w] = [{"def","labels","genus","loc":{dom: [...]}, "locw":{dom: [...]}}]。
    ents_by_dom: {dom: [物の語]}。たどる語は場面ごとに集める（一部・場所の段の決まりが場面ごとに違うため）。"""
    L = {}
    def senses(w):
        if w in L:
            return L[w]
        key = lookup(WKD, w)
        ss = WKD.get(key, [])[:MAX_DEFS_MID] if key else []
        rows = []
        for d, lb in ss:
            t = re.sub(r"\([^)]*\)", " ", d).strip()
            doc = nlp(t[:200])
            rows.append({"def": d, "labels": lb, "genus": genus(d, nlp, doc),
                         "loc": {dm: loc_targets(d, nlp, dm, False, doc) for dm in "PGD"},
                         "locw": {dm: loc_targets(d, nlp, dm, True, doc) for dm in wide_doms}})
        L[w] = rows
        return rows
    for dom, ents in ents_by_dom.items():
        front = set(ents)
        for w in ents:
            senses(w)
        for depth in range(DEPTH):
            nxt = set()
            for w in front:
                for x in L[w]:
                    tg = ([x["genus"]] if x["genus"] else []) + x["loc"][dom] + (x["locw"].get(dom, []) if dom in wide_doms else [])
                    for g in tg:
                        if g not in L:
                            senses(g)
                        nxt.add(g)
            log(f"{dom} 段{depth + 1}: 語の合計 {len(L)}")
            front = nxt
    return L


def next_of(L, w, k, dom, wide=False):
    """(w,k) から出る段：[(g, 'isa'|'loc')]"""
    x = L[w][k]
    out = []
    if x["genus"] and x["genus"] != w and L.get(x["genus"]):
        out.append((x["genus"], "isa"))
    locs = x["locw"].get(dom, []) if wide else x["loc"][dom]
    for g in locs:
        if g != w and L.get(g) and (g, "isa") not in out:
            out.append((g, "loc"))
    return out


def chain_nodes(L, x, dom, method, choice=None, wide=False):
    """物の語 x から3段までの点（語, 意味の番号）と段 ((w,k),(g,j),種類)。
    D：途中の語の意味は choice[(w,k,g)]（Gemmaの選び）。B：印のない最初の意味。物の語は全部の意味。"""
    nodes, links = set(), set()
    front = [(x, k) for k in range(len(L.get(x, [])))]
    nodes.update(front)
    for _ in range(DEPTH):
        nxt = []
        for (w, k) in front:
            for g, typ in next_of(L, w, k, dom, wide):
                if len(L[g]) == 1:
                    j = 0
                elif method == "B":
                    j = first_good(L[g])
                else:
                    j = (choice or {}).get((w, k, g), 0)
                links.add(((w, k), (g, j), typ))
                if (g, j) not in nodes:
                    nodes.add((g, j)); nxt.append((g, j))
        front = nxt
    return nodes, links


def choice_prompt(w, d, g, cands):
    opts = "\n".join(f"{i + 1}. {c}" for i, c in enumerate(cands))
    return (f"Dictionary definition of \"{w}\": {d}\n\nIn this definition, which meaning of the word \"{g}\" is used?\n"
            f"{opts}\n\nAnswer with the number only.")


def premise(w, d):
    return f"\"{w}\" means: {d}"


cap = lambda s: s[:1].upper() + s[1:]


def step_hyp(w, g, typ, dom):
    if typ == "isa":
        return f"{cap(w)} is a type of {g}."
    return {"P": f"{cap(w)} is a part of {g}.", "G": f"{cap(w)} is located in {g}.", "D": f"{cap(w)} affects the {g}."}[dom]


def arrive_hyps(w, cls, dom, meta):
    """点 w が分類 cls に「着く」かを問う文（1つか2つ）"""
    if dom == "P":
        return [f"{cap(w)} is a part of {meta['wholes'][cls]}."]
    if dom == "G":
        return [f"{w} is located in {meta['regions'][cls]}."]
    lb = meta["labels"][cls]
    hs = [f"{cap(w)} is {lb['art']} {lb['name']}."]
    if lb["system"]:
        hs.append(f"{cap(w)} is a part of the {lb['system']}.")
    return hs


def gemma_q(stmt):
    return f"Is the following statement true? Answer Yes or No only.\nStatement: {stmt}"   # E4〜E9と同じ聞き方

In [ ]:
META = json.loads(r'''{"wholes": {"body": "a human body", "head": "a head", "face": "a face", "hand": "a hand", "foot": "a foot", "eye": "an eye", "mouth": "a mouth", "arm": "an arm", "leg": "a leg", "tooth": "a tooth", "bird": "a bird", "fish": "a fish", "horse": "a horse", "tree": "a tree", "flower": "a flower", "car": "a car", "bicycle": "a bicycle", "airplane": "an airplane", "ship": "a ship", "boat": "a boat", "house": "a house", "church": "a church", "computer": "a computer", "watch": "a watch", "piano": "a piano", "gun": "a gun", "shoe": "a shoe", "shirt": "a shirt", "camera": "a camera", "knife": "a knife"}, "labels": {"cardio": {"name": "cardiovascular system disease", "art": "a", "system": "cardiovascular system"}, "resp": {"name": "respiratory system disease", "art": "a", "system": "respiratory system"}, "gi": {"name": "gastrointestinal system disease", "art": "a", "system": "digestive system"}, "nerv": {"name": "nervous system disease", "art": "a", "system": "nervous system"}, "skin": {"name": "skin disease", "art": "a", "system": "skin"}, "musc": {"name": "musculoskeletal system disease", "art": "a", "system": "musculoskeletal system"}, "urin": {"name": "urinary system disease", "art": "a", "system": "urinary system"}, "endo": {"name": "endocrine system disease", "art": "an", "system": "endocrine system"}, "immune": {"name": "immune system disease", "art": "an", "system": "immune system"}, "blood": {"name": "blood disease", "art": "a", "system": "blood"}, "repro": {"name": "reproductive system disease", "art": "a", "system": "reproductive system"}, "breast": {"name": "breast disease", "art": "a", "system": "breast"}, "chest": {"name": "thoracic disease", "art": "a", "system": "chest"}, "metab": {"name": "metabolic disease", "art": "a", "system": null}, "infect": {"name": "infectious disease", "art": "an", "system": null}, "tumor": {"name": "cancer or other tumor", "art": "a", "system": null}, "mental": {"name": "mental disorder", "art": "a", "system": null}, "genetic": {"name": "genetic disease", "art": "a", "system": null}, "bact": {"name": "bacterial infection", "art": "a", "system": null}, "viral": {"name": "viral infection", "art": "a", "system": null}, "fungal": {"name": "fungal infection", "art": "a", "system": null}, "parasit": {"name": "parasitic infection", "art": "a", "system": null}}, "regions": {"Micronesia": "Micronesia", "Eastern Europe": "Eastern Europe", "Western Europe": "Western Europe", "Melanesia": "Melanesia", "Middle Africa": "Middle Africa", "Eastern Asia": "Eastern Asia", "Northern America": "Northern America", "Southern Africa": "Southern Africa", "Western Asia": "Western Asia", "South-eastern Asia": "South-Eastern Asia", "Southern Asia": "Southern Asia", "Western Africa": "Western Africa", "Central Asia": "Central Asia", "South America": "South America", "Caribbean": "Caribbean", "Central America": "Central America", "Eastern Africa": "Eastern Africa", "Northern Europe": "Northern Europe", "Southern Europe": "Southern Europe", "Northern Africa": "Northern Africa", "Polynesia": "Polynesia", "Australia and New Zealand": "Australia and New Zealand"}, "date": "2026-11-11", "seed": 0}''')
ORDER = json.loads(r'''{"P": ["ab", "abdomen", "abdominal", "accelerator", "action", "air-intake", "airway", "anchor", "ankle", "annex", "anus", "aperture", "appendix", "arch", "argentine", "armpit", "arse", "articulation", "ass", "attic", "axle", "back", "backside", "balance", "bale", "ball", "band", "barb", "bark", "barker", "barrel", "bars", "base", "basic", "bath", "bathroom", "bathtub", "baton", "bay", "beak", "beard", "bearing", "bed", "bedroom", "behind", "belly", "belt", "bill", "bishop", "blade", "block", "bloodstream", "bloom", "blossom", "boiler", "bolt", "bomber", "bonnet", "bosom", "bottom", "bow", "bowel", "brain", "brain-stem", "brake", "brakes", "breast", "bridge", "brow", "brush", "buffer", "buffet", "bulb", "bum", "bumper", "buns", "bus", "butt", "butter", "buttocks", "cab", "cabin", "calf", "can", "cannon", "canopy", "cap", "capacitor", "capsule", "cartridge", "case", "casing", "cavity", "ceiling", "cement", "cervix", "chain", "chamber", "charcoal", "chassis", "cheek", "cherry", "chest", "chin", "chip", "choir", "choke", "chromosome", "clarence", "clip", "closure", "clothing", "clutch", "coat", "cock", "cockpit", "coffin", "collar", "compartment", "condo", "cone", "conjunction", "control", "cope", "coping", "corner", "cornerstone", "corolla", "corona", "cortex", "costa", "counter", "course", "court", "courtyard", "cross", "crotch", "crown", "crystal", "cuff", "cushion", "cusp", "cylinder", "dart", "dashboard", "deck", "den", "dial", "diaphragm", "digit", "dining-room", "disposable", "distributor", "dock", "door", "doorstep", "doorway", "drawer", "drawing", "duct", "ear", "earth", "elbow", "elevated", "elevator", "emergency", "enamel", "endocrine", "escape", "exhaust", "extension", "extremity", "eyeball", "eyebrow", "eyelid", "facial", "facing", "fall-board", "fan", "fanny", "feather", "feature", "femur", "fender", "fetus", "film", "fin", "finder", "finger", "firearm", "first", "flap", "flaps", "flood", "floor", "flooring", "floppy", "fluke", "fly", "fore", "forearm", "foreground", "forehead", "forget-me-not", "fork", "foundation", "foyer", "fraction", "frame", "funnel", "fuselage", "galley", "gap", "gas", "gather", "gathering", "gay-feather", "gear", "gearbox", "gene", "gorge", "graphics", "grille", "grip", "groin", "groundwork", "guitar", "gum", "gun-sight", "gut", "hair", "half-moon", "hall", "hammer", "handle", "hank", "harbour", "hardware", "header", "heart", "heat", "heating", "heel", "helm", "hemisphere", "high", "hinge", "hip", "hold", "hood", "hoof", "horn", "horseback", "hub", "hull", "ignition", "insulin", "intestine", "iris", "item", "jade", "jag", "jaw", "joint", "journal", "keel", "key", "keyboard", "kitchen", "knee", "knee-hi", "knuckle", "lace", "landing", "lap", "lash", "lat", "leaflet", "lens", "level", "library", "lid", "lien", "lift", "ligament", "limb", "liner", "lining", "lip", "litter-basket", "liver", "living-room", "load", "lobby", "lobe", "lock", "loft", "log", "low", "lure", "lymph", "magazine", "magneto", "mallet", "marina", "marrow", "matrix", "mattress", "memory", "middle", "milk", "missile", "mitt", "mixer", "monitor", "mortar", "mosque", "moth", "moustache", "movement", "muscle", "mustache", "muzzle", "nail", "nasal", "nave", "neck", "negative", "network", "neuron", "nib", "node", "nook", "nose", "nucleus", "oculus", "optic", "orb", "orbit", "organ", "os", "outfit", "pad", "palate", "palm", "pancreas", "pane", "parachute", "parlor", "parlour", "pavilion", "paw", "pawn", "pedal", "peel", "peg", "pen-and-ink", "peripheral", "pes", "petal", "picnic", "pier", "pixel", "placenta", "plate", "plug", "plume", "pocket", "pod", "point", "poll", "pool", "poop", "porch", "portfolio", "posterior", "pot", "potty", "processor", "produce", "pulp", "pump", "pupil", "quad", "quarry", "quarter", "quill", "radiator", "radius", "ray", "ready-mix", "ready-to-wear", "rear", "receptor", "register", "retina", "retreat", "reverse", "rib", "ride", "rim", "rod", "roe", "roof", "room", "root", "rose", "rotor", "row", "rudder", "rug", "rump", "saddle", "saliva", "sanctuary", "sash", "scooter", "scope", "screen", "screw", "seal", "seat", "seatbelt", "second", "self-starter", "set", "shaft", "shank", "she-goat", "sheet", "shelf", "shifter", "shin", "shock", "shoulder", "shower", "shroud", "shutter", "side", "six-gun", "skeleton", "skin", "skirt", "skull", "sleeve", "small", "snail", "snout", "socket", "sole", "soundtrack", "soy", "spar", "specific", "spike", "spit", "splash-guard", "spleen", "spoke", "squash", "stair", "staircase", "stairway", "starter", "stem", "step", "stern", "sticker", "stigma", "stock", "stomach", "stool", "stop", "storage", "store", "storey", "story", "stretcher", "string", "stripe", "stud", "study", "stump", "style", "sugar-bush", "sunshine-roof", "suspension", "swan", "tack", "tail", "tailor-made", "teeth", "temple", "tendon", "thermostat", "thigh", "third", "threshold", "throat", "throne", "throttle", "ticker", "time-ball", "toe", "toilet", "tongue", "top", "topping", "torso", "tract", "transmission", "tread", "trigger", "trunk", "tub", "tum", "tummy", "two", "universal", "upper", "upstairs", "valve", "vane", "vas", "vein", "vessel", "waist", "wall", "wardrobe", "ware", "web", "wheel", "wheeler", "wicket", "winch", "window", "windshield", "wing", "work-clothes", "wrapping", "wrist", "yoke"], "D": ["shrimp allergy", "aspirin allergy", "nickel allergic asthma", "cobalt allergic asthma", "mineral metabolism disease", "African tick-bite fever", "Japanese spotted fever", "inflammatory diarrhea", "Pontiac fever", "sensory system disease", "restless legs syndrome", "Usher syndrome", "yellow nail syndrome", "Costello syndrome", "Carney complex", "spinal polio", "Ross River fever", "West syndrome", "inflammatory bowel disease", "ADULT syndrome", "triple-A syndrome", "respiratory system cancer", "Rh deficiency syndrome", "Arts syndrome", "fetal alcohol syndrome", "female breast cancer", "mixed cerebral palsy", "small cell carcinoma", "cell type cancer", "anal canal cancer", "alcohol dependence", "mantle cell lymphoma", "marginal zone lymphoma", "peripheral T-cell lymphoma", "ARC syndrome", "short QT syndrome", "artery disease", "peripheral artery disease", "sleep apnea", "renal fibrosis", "ovarian melanoma", "stroke", "withdrawal disorder", "Good syndrome", "dropped head syndrome", "specific developmental disorder", "pervasive developmental disorder", "autism spectrum disorder", "atypical autism", "sexual health disorder", "writing disorder", "gluten allergy", "lymphoma", "Pearson syndrome", "triple-negative breast cancer", "immune system cancer", "cartilage cancer", "alpha chain disease", "mu chain disease", "pain disorder", "bipolar ll disorder", "Timothy syndrome", "Carpenter syndrome", "UV-sensitive syndrome", "specific language impairment", "Mast syndrome", "cold-induced sweating syndrome", "cardiac arrest", "Cole-Carpenter syndrome", "EAST syndrome", "Perry syndrome", "Pitt-Hopkins syndrome", "chicken egg allergy", "shellfish allergy", "respiratory allergy", "pollen allergy", "Timothy grass allergy", "drug allergy", "metal allergy", "fruit allergy", "cherry allergy", "Indian plum allergy", "orange allergy", "melon allergy", "peach allergy", "tomato allergy", "fish allergy", "Atlantic cod allergy", "Atlantic salmon allergy", "rainbow trout allergy", "cow milk allergy", "goat milk allergy", "brown shrimp allergy", "crab allergy", "latex allergy", "Char syndrome", "spinal disease", "abdominal obesity-metabolic syndrome", "Peters anomaly", "hand-foot-genital syndrome", "Wilson-Turner syndrome", "juvenile-onset Parkinson's disease", "Norman-Roberts syndrome", "White-Sutton syndrome", "absence epilepsy", "drug-induced hearing loss", "celery allergy", "Watson syndrome", "Mitchell syndrome", "ring chromosome syndrome", "Ramon syndrome", "alcohol-associated liver disease", "alpha-gal syndrome", "Davis-Wells syndrome", "HELIX syndrome", "lung-brain developmental disorder", "muscular disease", "bone disease", "bone development disease", "bone deterioration disease", "X-linked dominant disease", "bone structure disease", "Worth syndrome", "intestinal pseudo-obstruction", "PAPA syndrome", "Sweeney-Cox syndrome", "tongue carcinoma", "NK cell deficiency", "gene duplication disease", "Sweet syndrome", "urinary tract infection", "intermittent asthma", "persistent mild asthma", "persistent moderate asthma", "childhood-onset asthma", "adult-onset severe asthma", "environmentally induced asthma", "occupational asthma", "aspirin-induced respiratory disease", "near-fatal asthma", "persistent severe asthma", "thunderstorm triggered asthma", "mild cognitive impairment", "long COVID", "hot water epilepsy", "injection anthrax", "graft-versus-host disease", "breast implant illness", "Becker disease", "semantic dementia", "sickle cell disease", "large B-cell lymphoma", "auto-brewery syndrome", "bladder fermentation syndrome", "RIDDLE syndrome", "African iron overload", "Scott syndrome", "Jackson-Weiss syndrome", "Floating-Harbor syndrome", "scalp-ear-nipple syndrome", "stiff skin syndrome", "Gillespie syndrome", "C syndrome", "COACH syndrome", "Cohen syndrome", "Freeman-Sheldon syndrome", "DOORS syndrome", "Saul-Wilson syndrome", "MEND syndrome", "McLeod syndrome", "CHIME syndrome", "intestinal infectious disease", "thyroid lymphoma", "migraine with aura", "eye accommodation disease", "sleeping sickness", "dry eye syndrome", "intestinal cancer", "leprosy", "prostate cancer", "prostate carcinoma", "prostate lymphoma", "viral meningitis", "chronic meningitis", "aggressive NK-cell leukemia", "chronic leukemia", "common cold", "diaphragm disease", "beach ear", "viral pneumonia", "calcium metabolism disease", "chronic tic disorder", "glucose intolerance", "blind loop syndrome", "lymph node cancer", "Alzheimer's disease", "portal hypertension", "hypertension", "nasal cavity cancer", "malignant essential hypertension", "essential hypertension", "Edwards syndrome", "partial third-nerve palsy", "total third-nerve palsy", "benign essential hypertension", "obsessive-compulsive personality disorder", "paranoid personality disorder", "antisocial personality disorder", "tongue disease", "ovarian disease", "urinary bladder cancer", "Q fever", "spotted fever", "secondary hypertension", "respiratory failure", "pulp degeneration", "opioid abuse", "plethora of newborn", "social phobia", "cat-scratch disease", "tick paralysis", "bladder calculus", "functional diarrhea", "heart disease", "brain compression", "Frey syndrome", "blue color blindness", "heart cancer", "Lyme disease", "iron deficiency anemia", "absolute glaucoma", "protein-energy malnutrition", "bladder neck cancer", "bladder dome cancer", "bladder lymphoma", "visual epilepsy", "coronary artery anomaly", "vaginal cancer", "morbid obesity", "drug-induced mental disorder", "vaginal disease", "factor VIII deficiency", "low compliance bladder", "descending colon cancer", "cascade stomach", "paranoid schizophrenia", "cough variant asthma", "male infertility", "alcoholic hepatitis", "Graves' disease", "malaria", "complex partial epilepsy", "pathological gambling", "leukemia", "intermittent explosive disorder", "acute leukemia", "locked-in syndrome", "migraine without aura", "thyroid crisis", "autistic disorder", "Miller Fisher syndrome", "conduct disorder", "shipyard eye", "transient global amnesia", "dementia", "protein-deficiency anemia", "benign secondary hypertension", "gout", "brain cancer", "lung cancer", "Rift Valley fever", "intestinal tuberculosis", "pedophilia", "Kawasaki disease", "hairy tongue", "twin-to-twin transfusion syndrome", "frontal sinus cancer", "prostate calculus", "malignant secondary hypertension", "sweat gland disease", "sick sinus syndrome", "red color blindness", "acquired color blindness", "visual pathway disease", "facial paralysis", "bladder neck obstruction", "chronic orbital inflammation", "POEMS syndrome", "bipolar I disorder", "anus cancer", "toxic shock syndrome", "blue toe syndrome", "ovarian dysfunction", "pulp erosion", "spinal cord lymphoma", "spinal cord melanoma", "Down syndrome", "fragile X syndrome", "blindness", "generalized anxiety disorder", "mixed malaria", "Parkinson's disease", "estrogen excess", "farmer's lung", "dumping syndrome", "Foster-Kennedy syndrome", "Binder syndrome", "major depressive disorder", "Weaver syndrome", "cholera", "personality disorder", "cognitive disorder", "pulmonary systemic sclerosis", "depressive disorder", "breast cancer", "male breast cancer", "cancer", "skeletal tuberculosis", "congenital heart disease", "glaucoma", "drug psychosis", "vascular cancer", "facial nerve disease", "cardiovascular cancer", "conversion disorder", "vascular disease", "thyroid cancer", "pancreatic cancer", "generalized epilepsy", "bone cancer", "jaw cancer", "sexual dysfunction", "hepatitis C", "viral hepatitis", "melanoma", "reproductive organ cancer", "Laurence-Moon syndrome", "cerebral palsy", "anxiety disorder", "hepatitis B", "drug-induced hepatitis", "post-traumatic stress disorder", "transient arthritis", "sweat gland cancer", "heel spur", "B cell deficiency", "ovarian lymphoma", "nasal cavity disease", "ascending colon cancer", "colon cancer", "factor VII deficiency", "factor V deficiency", "factor X deficiency", "factor XI deficiency", "focal epilepsy", "motor neuron disease", "iron metabolism disease", "anemia", "pupil disease", "renal artery disease", "ovarian cancer", "protein S deficiency", "psychotic disorder", "phosphorus metabolism disease", "enamel erosion", "alcohol-induced mental disorder", "reflex epilepsy", "morphine dependence", "pancreas disease", "transverse colon cancer", "kidney cancer", "transitional cell carcinoma", "mushroom workers' lung", "sick building syndrome", "Bloom syndrome", "bone carcinoma", "tic disorder", "Dandy-Walker syndrome", "middle lobe syndrome", "nose disease", "asthma", "hairy cell leukemia", "Newcastle disease", "eye carcinoma", "motion sickness", "inner ear disease", "pulmonary tuberculosis", "renal artery obstruction", "cystic kidney disease", "carbohydrate metabolic disorder", "germ cell cancer", "substance abuse", "acute kidney failure", "food allergy", "carcinoma", "nervous system cancer", "gastrointestinal system cancer", "gastric outlet obstruction", "anus disease", "lipid metabolism disorder", "lipid pneumonia", "bipolar disorder", "mood disorder", "partial motor epilepsy", "partial sensory epilepsy", "bone inflammation disease", "accessory nerve disease", "coronary artery disease", "peripheral vascular disease", "penis carcinoma", "uterine disease", "skin carcinoma", "breast carcinoma", "iris cancer", "Turner syndrome", "chronic wasting disease", "liver cancer", "uterine cancer", "bladder disease", "Leigh disease", "wheat allergy", "olfactory nerve disease", "childhood kidney cancer", "skin disease", "nutrition disease", "protein C deficiency", "vaginal discharge", "pulmonary fibrosis", "Coffin-Lowry syndrome", "external ear disease", "skull base cancer", "Lynch syndrome", "lung carcinoma", "tuberculosis", "urinary system cancer", "bladder carcinoma", "oral tuberculosis", "mouth disease", "gastrointestinal tuberculosis", "skeletal muscle cancer", "muscle cancer", "liver disease", "tick infestation", "nail disease", "orbital cancer", "primary syphilis", "secondary syphilis", "syphilis", "autoimmune disease", "articulation disorder", "glucose metabolism disease", "brain stem cancer", "hair disease", "smooth muscle cancer", "anal margin carcinoma", "Ebola virus disease", "burning mouth syndrome", "cervical cancer", "milk allergy", "egg allergy", "peanut allergy", "nut allergy", "hepatitis E", "dyslexia", "large cell carcinoma", "bile duct cancer", "prostate disease", "mongolian spot", "space motion sickness", "movement disease", "Colorado tick fever", "middle ear carcinoma", "anal carcinoma", "bone marrow cancer", "bone marrow disease", "neurotic disorder", "essential tremor", "thyroid gland disease", "adjustment disorder", "middle ear cancer", "middle ear disease", "nutritional deficiency disease", "aortic disease", "urinary tract obstruction", "intestinal disease", "eyelid disease", "Roberts syndrome", "dental pulp disease", "sleep disorder", "schizophrenia", "stomach carcinoma", "pneumonia", "kidney disease", "Ferguson-Smith tumor", "spinal cancer", "eye disease", "large intestine cancer", "retinal disease", "Werner syndrome", "visual cortex disease", "nerve compression syndrome", "lung clear cell-sugar-tumor", "cerebral lymphoma", "gray zone lymphoma", "childhood lymphoma", "breast lymphoma", "panic disorder", "specific phobia", "animal phobia", "appendix disease", "pulmonary artery disease", "AIDS phobia", "flying phobia", "acute stress disorder", "anal canal carcinoma", "aortic valve disease", "blood protein disease", "complement deficiency", "inflammatory breast carcinoma", "migraine", "pulmonary hypertension", "inherited metabolic disorder", "muscle tissue disease", "spleen cancer", "bilateral breast cancer", "bone lymphoma", "compartment syndrome", "liver carcinoma", "skin tag", "combat disorder", "pancreatic cholera", "anthrax disease", "inhibited female orgasm", "bladder tuberculosis", "stomach disease", "retinal cell cancer", "retina lymphoma", "Coats disease", "delusional disorder", "renal infectious disease", "chronic kidney disease", "female breast carcinoma", "sporadic breast cancer", "septic arthritis", "auditory system cancer", "intestinal obstruction", "influenza", "arthritis", "hereditary night blindness", "night blindness", "lung disease", "collagen disease", "lip cancer", "soft palate cancer", "scarlet fever", "upper gum cancer", "oral cavity cancer", "measles", "substance-induced psychosis", "tongue cancer", "vein disease", "eating disorder", "genital herpes", "vascular dementia", "bacterial pneumonia", "lower lip cancer", "skin melanoma", "learning disability", "metal metabolism disorder", "liver lymphoma", "hard palate cancer", "speech disorder", "Ramsay Hunt syndrome", "mast-cell leukemia", "lip disease", "language disorder", "orbital disease", "Taylor's syndrome", "brain disease", "intrinsic asthma", "lipid storage disease", "bacterial meningitis", "meningitis", "cannabis abuse", "plasma cell leukemia", "vascular skin disease", "cervical incompetence", "yellow fever", "renal tuberculosis", "heart aneurysm", "bowel dysfunction", "eye degenerative disease", "cardiovascular syphilis", "developmental coordination disorder", "null-cell leukemia", "obesity", "substance dependence", "drug dependence", "heroin dependence", "orbit lymphoma"], "G": ["United Arab Emirates", "Dubai", "Al Muteena", "Mirdif", "Al Waheda", "Afghanistan", "Kabul", "Jabal os Saraj", "Ghazni", "Kunduz", "Albania", "Tirana", "Pogradec", "Librazhd", "Rrashbull", "Angola", "Mulenvos", "Caluquembe", "Nzagi", "Calulo", "Argentina", "Buenos Aires", "General Belgrano", "Veinticinco de Mayo", "El Palomar", "Austria", "Graz", "Innere Stadt", "Gries", "Favoriten", "Australia", "Brisbane", "Hoppers Crossing", "Umina Beach", "Bunbury", "Bosnia and Herzegovina", "Sarajevo", "Lukavac", "Prijedor", "Zenica", "Bangladesh", "Dhaka", "Khulna", "Chattogram", "Kushtia", "Belgium", "Brussels", "Kontich", "Ekeren", "Boussu", "Burkina Faso", "Ouagadougou", "Boromo", "Yako", "Bittou", "Bulgaria", "Sofia", "Rakovski", "Kyustendil", "Nova Zagora", "Burundi", "Bujumbura", "Zanandore", "Kirundo", "Vyanda", "Benin", "Cotonou", "Abomey", "Banikoara", "Adjarra", "Brunei", "Bandar Seri Begawan", "Mentiri", "Kuala Belait", "Serasa", "Bolivia", "Santa Cruz de la Sierra", "Huanuni", "Achocalla", "Sacaba", "Brazil", "Rio de Janeiro", "Sousa", "Ruy Barbosa", "Quatro Barras", "Botswana", "Gaborone", "Letlhakane", "Lobatse", "Mahalapye", "Belarus", "Minsk", "Novopolotsk", "Kalodzishchy", "Zhlobin", "Belize", "Belize City", "Orange Walk", "Belmopan", "San Ignacio", "Canada", "Toronto", "Mount Olive-Silverstone-Jamestown", "Hespeler", "McKenzie Towne", "Democratic Republic of the Congo", "Kinshasa", "Kikwit", "Katanda", "Lukula", "Central African Republic", "Bangui", "Sibut", "Paoua", "Bouar", "Republic of the Congo", "Brazzaville", "Makoua", "Loudima", "Mossendjo", "Switzerland", "Basel", "Vevey", "Luzern", "Bulle", "Ivory Coast", "Abidjan", "Aboka", "Moussadougou", "Dabou", "Chile", "Puente Alto", "Pozo Almonte", "Padre Las Casas", "Coihueco", "Cameroon", "Douala", "Nkoteng", "Kumba", "Okoa", "China", "Shanghai", "Wuzhou", "Kaitong", "Guangshun", "Colombia", "Cali", "Quimbaya", "Lorica", "Anserma", "Costa Rica", "Alajuela", "Chacarita", "Puntarenas", "San Rafael Arriba", "Cuba", "Havana", "Abreus", "Nuevitas", "Yara", "Cabo Verde", "Praia", "Espargos", "Mindelo", "Assomada", "Czechia", "Prague", "Chrudim", "Znojmo", "Trutnov", "Germany", "Berlin", "Bergisch Gladbach", "Northeim", "Seevetal", "Djibouti", "Balbala", "Arta", "Obock", "Tadjoura", "Denmark", "Copenhagen", "Kolding", "Ringsted", "Kalundborg", "Dominican Republic", "Santiago de los Caballeros", "Santo Domingo Oeste", "Neiba", "San Fernando de Monte Cristi", "Algeria", "Algiers", "Arbatache", "Souk Ahras", "Ouled Mimoun", "Ecuador", "Quito", "Riobamba", "Pelileo", "Otavalo", "Estonia", "Tallinn", "Kesklinn", "Narva", "Pirita", "Eritrea", "Asmara", "Barentu", "Mendefera", "Keren", "Spain", "Palma", "La Solana", "Leioa", "Pinto", "Ethiopia", "Addis Ababa", "Burayu", "Lasoano", "Indabaguna", "Finland", "Helsinki", "Espoonlahti", "Porvoo", "Varkaus", "Fiji", "Nasinu", "Lami", "Sigatoka", "Suva", "France", "Marseille", "Champigny-sur-Marne", "Cergy", "Maurepas", "Gabon", "Libreville", "Koulamoutou", "Port-Gentil", "Owendo", "United Kingdom", "Glasgow", "Berkhamsted", "Rawtenstall", "Newcastle under Lyme", "French Guiana", "Cayenne", "Saint-Laurent-du-Maroni", "Matoury", "Kourou", "Ghana", "Kumasi", "Nsoatre", "Wa", "Taifa", "Gambia", "Serekunda", "Latri Sabiji", "Farato", "Banjul", "Guinea", "Conakry", "Siguiri", "Banko", "Coyah", "Guadeloupe", "Les Abymes", "Sainte-Anne", "Sainte-Rose", "Capesterre-Belle-Eau", "Equatorial Guinea", "Bata", "Ebebiyin", "Ciudad de la Paz", "Malabo", "Greece", "Piraeus", "Corfu", "Rethymno", "Chios", "Guatemala", "Guatemala City", "Ciudad Vieja", "Sumpango", "Sanarate", "Guam", "Dededo Village", "Tamuning", "Yigo Village", "Tamuning-Tumon-Harmon Village", "Guinea-Bissau", "Bissau", "Xitole", "Sintra-Nema", "Contumo", "Hong Kong", "New Territories", "Braemar Hill", "Wong Tai Sin", "To Kwa Wan", "Honduras", "Tegucigalpa", "Teupasenti", "Langue", "Sulaco", "Croatia", "Zagreb", "Rijeka", "Slavonski Brod", "Split", "Haiti", "Port-au-Prince", "Grangwav", "Kenscoff", "Dessalines", "Hungary", "Budapest", "Szentendre", "Karcag", "Dabas", "Ireland", "South Dublin", "Malahide", "Sandyford", "Galway", "Israel", "Jerusalem", "Ness Ziona", "Ar Rayna", "Qiryat Ata", "India", "Mumbai", "Vettavalam", "Saoner", "Sevilimedu", "Iraq", "Baghdad", "Dihok", "Al Hillah", "Al-'Alam", "Iran", "Tehran", "Nowshahr", "Khorramshahr", "Surian", "Italy", "Milan", "Sora", "Grugliasco", "Eppan", "Jamaica", "New Kingston", "Portmore", "Spanish Town", "Montego Bay", "Jordan", "Amman", "Al Jubayhah", "Kurayyimah", "Aqaba", "Japan", "Tokyo", "Kamiiso", "Wakimachi", "Kushimoto", "Kenya", "Nairobi", "Wajir", "Takaba", "Makindu", "Kyrgyzstan", "Bishkek", "Karakol", "Kant", "Manas", "Cambodia", "Phnom Penh", "Sihanoukville", "Sisophon", "Kandoeng", "North Korea", "Pyongyang", "Chunggang", "Namyang", "Ayang-ni", "South Korea", "Seoul", "Gwangmyeong", "Ara-dong", "Seosan", "Kuwait", "Kuwait City", "Ar Riqqah", "Ad Dasmah", "Al Manqaf", "Laos", "Vientiane", "Muang Xay", "Vang Vieng", "Savannakhet", "Lebanon", "Beirut", "Baalbek", "Bsharri", "Byblos", "Sri Lanka", "Dehiwala-Mount Lavinia", "Kolonnawa", "Kurunegala", "Kataragama", "Liberia", "Gbarnga", "Foya Kamara", "Kakata", "Pleebo City", "Lesotho", "Maseru", "Butha-Buthe", "Quthing", "Teyateyaneng", "Lithuania", "Vilnius", "Alytus", "Radviliskis", "Jonava", "Latvia", "Riga", "Jelgava", "Ogre", "Salaspils", "Libya", "Benghazi", "Zintan", "Al Burayqah", "Zliten", "Morocco", "Rabat", "Ain El Aouda", "Lahraouyine", "Ben Taieb", "Moldova", "Chisinau", "Bilicenii Vechi", "Orhei", "Ungheni", "Montenegro", "Podgorica", "Pljevlja", "Bijelo Polje", "Budva", "Madagascar", "Antananarivo", "Imerimandroso", "Moramanga", "Imerintsiatosika", "North Macedonia", "Skopje", "Bogovinje", "Tearce", "Radovis", "Mali", "Bamako", "Dougabougou", "Bougouni", "Fana", "Myanmar", "Yangon", "Dawei", "Kalemyo", "Lawksawk", "Mongolia", "Ulan Bator", "Saynshand", "Bulgan", "Bogd", "Macao", "Taipa", "Zhuojiacun", "Lai Chi Van", "Luhuan", "Martinique", "Fort-de-France", "Le Robert", "Le Lamentin", "Ducos", "Mauritania", "Nouakchott", "Akjoujt", "Ayoun El Atrous", "Atar", "Malta", "Birkirkara", "Valletta", "Sliema", "Mosta", "Mauritius", "Port Louis", "Centre de Flacq", "Triolet", "Saint Pierre", "Malawi", "Lilongwe", "Nkhotakota", "Nsanje", "Mulanje", "Mexico", "Mexico City", "Tula de Allende", "Celaya", "Santiago Ixcuintla", "Malaysia", "Kuala Lumpur", "Gelang Patah", "Julau", "Sungai Udang", "Mozambique", "Maputo", "Matola", "Quelimane", "Vilankulo", "Namibia", "Windhoek", "Tsumeb", "Katutura", "Walvis Bay", "Niger", "Niamey", "Dosso", "Guidan Roumdji", "Madarounfa", "Nigeria", "Kano", "Uga", "Okrika", "Jebba", "Nicaragua", "Managua", "La Paz Centro", "Diriamba", "Bluefields", "The Netherlands", "The Hague", "Nijverdal", "Heemstede", "Amsterdam-Zuidoost", "Norway", "Oslo", "Trondheim", "Fredrikstad", "Porsgrunn", "Nepal", "Kathmandu", "Gaur", "Hetauda", "Janakpur", "New Zealand", "Auckland", "Whakatane", "Whanganui", "Manukau City", "Oman", "Muscat", "As Suwayq", "Seeb", "Bidbid", "Panama", "San Miguelito", "Parque Lefevre", "Santiago de Veraguas", "La Cabima", "Peru", "Callao", "Yurimaguas", "Atalaya", "Sechura", "Philippines", "Quezon City", "Tayabas", "Sasmuan", "Panabo", "Pakistan", "Lahore", "Ghotki", "Mananwala", "New Mirpur City", "Poland", "Warsaw", "Przasnysz", "Legionowo", "Wola", "Puerto Rico", "Ponce", "Fajardo", "Vega Baja", "Arecibo", "Palestinian Territory", "East Jerusalem", "Ramallah", "Nablus", "Battir", "Portugal", "Lisbon", "Albufeira", "Afurada de Baixo", "Amora", "Paraguay", "Ciudad del Este", "Nemby", "Presidente Franco", "Villa Elisa", "Qatar", "Doha", "Lusail", "Al Wakrah", "An Najmah", "Reunion", "Saint-Paul", "La Possession", "Le Port", "Le Tampon", "Romania", "Bucharest", "Voluntari", "Breaza", "Slatina", "Serbia", "Belgrade", "Vranje", "Loznica", "Smederevo", "Rwanda", "Kigali", "Bushenge", "Ngororero", "Nyamata", "Saudi Arabia", "Jeddah", "Baljurashi", "Turubah", "Ranyah", "Solomon Islands", "Honiara", "Nggosi", "Tandai", "Malango", "Sudan", "Khartoum", "Wad Medani", "Kuraymah", "Ghubaysh", "Sweden", "Stockholm", "Boo", "Trelleborg", "Hudiksvall", "Singapore", "Ulu Bedok", "Bukit Batok New Town", "Thomson", "Choa Chu Kang New Town", "Slovenia", "Ljubljana", "Ptuj", "Velenje", "Celje", "Slovakia", "Bratislava", "Zvolen", "Snina", "Galanta", "Sierra Leone", "Freetown", "Njala", "Koidu", "Makeni", "Senegal", "Dakar", "Tambacounda", "Matam", "Thilogne", "Somalia", "Mogadishu", "Hargeysa", "Dhamas", "Jawhar", "Suriname", "Paramaribo", "Welgelegen", "Flora", "Munder Buiten", "South Sudan", "Juba", "Yambio", "Kuacjok", "Rumbek", "El Salvador", "San Salvador", "Cojutepeque", "Quezaltepeque", "Acajutla", "Syria", "Aleppo", "An Nabk", "Manbij", "Ar Raqqah", "Chad", "N'Djamena", "Fada", "Fianga", "Bokoro", "Togo", "Kara", "Dapaong", "Tchamba", "Niamtougou", "Thailand", "Bangkok", "Lak Si", "Bang Na", "Bang Rakam", "Tajikistan", "Dushanbe", "Khorugh", "Yovon", "Danghara", "Turkmenistan", "Ashgabat", "Gyzylarbat", "Magdanly", "Seydi", "Tunisia", "Tunis", "Maatmeur", "Midoun", "Metlaoui", "Trinidad and Tobago", "Chaguanas", "Mon Repos", "Laventille", "Port of Spain", "Tanzania", "Dar es Salaam", "Makongolosi", "Babati", "Ilula", "Ukraine", "Kyiv", "Otamanivka", "Bortnychi", "Desnyanskyi", "Uganda", "Kampala", "Buyale", "Koboko", "Kireka", "United States", "New York City", "Baileys Crossroads", "East Point", "Greenburgh", "Uruguay", "Montevideo", "Las Piedras", "Colonia del Sacramento", "Durazno", "Uzbekistan", "Tashkent", "Yangiyer", "Kirguli", "Payshamba Shahri", "Venezuela", "Caracas", "Biruaca", "La Paragua", "San Juan de los Morros", "Vietnam", "Ho Chi Minh City", "Sa Dec", "Hoi An", "Hanoi", "Yemen", "Sanaa", "Ibb", "Aden", "Sa'dah", "Mayotte", "Mamoudzou", "Koungou", "Dzaoudzi", "Labattoir", "South Africa", "Johannesburg", "Verena", "Marikana", "Clydesdale", "Zambia", "Lusaka", "Isoka", "Palabana", "Sinda", "Zimbabwe", "Harare", "Mvurwi", "Chipinge", "Redcliff"]}''')
CORDER = json.loads(r'''{"P": ["body", "head", "face", "hand", "foot", "eye", "mouth", "arm", "leg", "tooth", "bird", "fish", "horse", "tree", "flower", "car", "bicycle", "airplane", "ship", "boat", "house", "church", "computer", "watch", "piano", "gun", "shoe", "shirt", "camera", "knife"], "D": ["cardio", "resp", "gi", "nerv", "skin", "musc", "urin", "endo", "immune", "blood", "repro", "breast", "chest", "metab", "infect", "tumor", "mental", "genetic", "bact", "viral", "fungal", "parasit"], "G": ["Australia and New Zealand", "Caribbean", "Central America", "Central Asia", "Eastern Africa", "Eastern Asia", "Eastern Europe", "Melanesia", "Micronesia", "Middle Africa", "Northern Africa", "Northern America", "Northern Europe", "Polynesia", "South America", "South-eastern Asia", "Southern Africa", "Southern Asia", "Southern Europe", "Western Africa", "Western Asia", "Western Europe"]}''')
def remake(order, corder, meta):
    """ノートの中で主張の文を作り直す（e10_data.py と同じ書き方）。"""
    cap = lambda s: s[:1].upper() + s[1:]
    art = lambda w: ("an " if w[:1].lower() in "aeiou" else "a ") + w
    out = []
    for x in order["P"]:
        for W in corder["P"]:
            ph = meta["wholes"][W]
            out.append(["P", x, W, f"{cap(x)} is a part of {ph}.", f"{cap(ph.split(' ', 1)[1])} is a part of {art(x)}."])
    for x in order["D"]:
        for l in corder["D"]:
            lb = meta["labels"][l]
            out.append(["D", x, l, f"{cap(x)} is {lb['art']} {lb['name']}.", None])
    for x in order["G"]:
        for g in corder["G"]:
            out.append(["G", x, g, f"{x} is located in {meta['regions'][g]}.", None])
    return out
ITEMS = remake(ORDER, CORDER, META)
assert hashlib.md5(json.dumps(ITEMS).encode()).hexdigest()[:10] == 'f5566e013e', '主張の作り直しが合わない'
ENTS = {d: sorted({it[1] for it in ITEMS if it[0] == d}) for d in 'PDG'}
CLASSES = {d: sorted({it[2] for it in ITEMS if it[0] == d}) for d in 'PDG'}
print(len(ITEMS), '主張', {d: (len(ENTS[d]), len(CLASSES[d])) for d in 'PDG'})
print('確認の印:', hashlib.md5(json.dumps(ITEMS).encode()).hexdigest()[:10])

## 1. Gemmaに全部の主張を「はい／いいえ」で聞く（GPU、10〜15分）

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
G = load_gz("E10_gemma")
if G is None:
    tok = AutoTokenizer.from_pretrained(GEMMA_ID, token=userdata.get("HF_TOKEN"), padding_side="left")
    gm = AutoModelForCausalLM.from_pretrained(GEMMA_ID, token=userdata.get("HF_TOKEN"), torch_dtype=torch.float16, device_map="cuda").eval()
    YES = tok.encode("Yes", add_special_tokens=False)[0]; NO = tok.encode("No", add_special_tokens=False)[0]
    Q = [it[3] for it in ITEMS] + [it[4] for it in ITEMS if it[4]]
    def ask(stmts, B=64):
        out = []; t = time.time()
        for i in range(0, len(stmts), B):
            ps = [tok.apply_chat_template([{"role": "user", "content": gemma_q(s)}], tokenize=False, add_generation_prompt=True) for s in stmts[i:i + B]]
            enc = tok(ps, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
            with torch.no_grad():
                last = gm(**enc, logits_to_keep=1).logits[:, -1, :].float()
            out += (last[:, YES] - last[:, NO]).cpu().numpy().round(3).tolist()
            if (i // B) % 100 == 0: print(f"{i}/{len(stmts)}  {time.time() - t:.0f}秒")
        return out
    lg = ask(Q)
    n = len(ITEMS)
    rev = iter(lg[n:])
    G = {"logit": lg[:n], "rev": [next(rev) if it[4] else None for it in ITEMS], "model": GEMMA_ID}
    save_gz(G, "E10_gemma")
    del gm; torch.cuda.empty_cache()
lg = np.array(G["logit"])
for d in "PDG":
    m = np.array([it[0] == d for it in ITEMS]); print(d, "はいの割合", round(float((lg[m] > 0).mean()), 3))

## 2. Wiktionaryのダンプを取って、名詞と固有名詞を読む（30〜50分、GPUは使わない）

In [ ]:
wk_path = f"{OUT}/E10_wiktionary_senses.json.gz"
if os.path.exists(wk_path):
    with gzip.open(wk_path, "rt", encoding="utf-8") as f: WKD = json.load(f)
    print("ドライブから読み込み:", len(WKD))
else:
    dump = "/content/enwiktionary.xml.bz2"
    subprocess.run(["wget", "-c", "-q", "--show-progress", "-O", dump, DUMP_URL], check=True)
    print("取れた大きさ:", round(os.path.getsize(dump) / 1e9, 2), "GB")
    t = time.time()
    with bz2.open(dump, "rb") as f:
        WKD = parse_dump(f)
    print("読み終わり:", len(WKD), "語", f"{time.time() - t:.0f}秒")
    with gzip.open(wk_path, "wt", encoding="utf-8") as f: json.dump(WKD, f, ensure_ascii=False)
for w in ["pedal", "pneumonia", "Osaka", "Japan", "Maputo"]:
    k = lookup(WKD, w); print(w, "→", [d for d, lb in WKD.get(k, [])[:3]] if k else "なし")
for d in "PDG":
    print(d, "説明文のある物の割合:", round(float(np.mean([lookup(WKD, w) is not None for w in ENTS[d]])), 3))

## 3. 語の意味の一覧（鎖に出てくる語を集める、5〜10分）

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm", disable=["ner"])
L = load_gz("E10_lexicon")
if L is None:
    t = time.time()
    L = build_lexicon(ENTS, WKD, nlp)
    print("語:", len(L), f"{time.time() - t:.0f}秒")
    save_gz(L, "E10_lexicon")
del WKD
NEED = set()   # 意味を選ぶ必要のある段 (w,k,g)
def walk(dom, wide):
    for x in ENTS[dom]:
        front, seen = [(x, k) for k in range(len(L.get(x, [])))], set()
        for _ in range(DEPTH):
            nxt = []
            for (w, k) in front:
                for g, typ in next_of(L, w, k, dom, wide):
                    if len(L[g]) > 1: NEED.add((w, k, g))
                    for j in range(len(L[g])):
                        if (g, j) not in seen: seen.add((g, j)); nxt.append((g, j))
            front = nxt
for d in "PDG": walk(d, False)
walk("P", True)
NEED = sorted(NEED)
print("意味を選ぶ必要のある段:", len(NEED))

## 4. 途中の語の意味をGemmaに選ばせる（GPU、15〜25分、途中の控えあり）

In [ ]:
CH = load_gz("E10_choice_D") or []
CHOICE_D = {(w, k, g): v for w, k, g, v in CH}
todo = [x for x in NEED if x not in CHOICE_D]
print("済み:", len(CHOICE_D), " 残り:", len(todo))
if todo:
    tok = AutoTokenizer.from_pretrained(GEMMA_ID, token=userdata.get("HF_TOKEN"), padding_side="left")
    gm = AutoModelForCausalLM.from_pretrained(GEMMA_ID, token=userdata.get("HF_TOKEN"), torch_dtype=torch.float16, device_map="cuda").eval()
    NUM = [tok.encode(str(i), add_special_tokens=False)[0] for i in range(1, MAX_DEFS_MID + 1)]
    t = time.time(); B = 16
    for i in range(0, len(todo), B):
        batch = todo[i:i + B]
        qs = [choice_prompt(w, L[w][k]["def"], g, [x["def"] for x in L[g]]) for w, k, g in batch]
        ps = [tok.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True) for q in qs]
        enc = tok(ps, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        with torch.no_grad():
            last = gm(**enc, logits_to_keep=1).logits[:, -1, :].float()
        for (w, k, g), row in zip(batch, last):
            CHOICE_D[(w, k, g)] = int(row[NUM[:len(L[g])]].argmax())
        if (i // B) % 200 == 0:
            print(f"{i}/{len(todo)}  {time.time() - t:.0f}秒")
            save_gz([[w, k, g, v] for (w, k, g), v in CHOICE_D.items()], "E10_choice_D")
    save_gz([[w, k, g, v] for (w, k, g), v in CHOICE_D.items()], "E10_choice_D")
    del gm; torch.cuda.empty_cache()
print("最初の意味以外を選んだ割合:", round(float(np.mean([v != 0 for v in CHOICE_D.values()])), 3))

## 5. 鎖を作り、NLIで判定する（GPU、60〜100分、途中の控えあり）

In [ ]:
VARIANTS = [("D", False), ("B", False)]
CHAINS = {}
for d in "PDG":
    for m, wide in VARIANTS + ([("D", True)] if d == "P" else []):
        tag = m + ("w" if wide else "")
        for x in ENTS[d]:
            nodes, links = chain_nodes(L, x, d, m, CHOICE_D, wide)
            CHAINS[(d, tag, x)] = (sorted(nodes), sorted(links))
STEPS = sorted({(d, a[0], a[1], b[0], typ) for (d, tag, x), (nd, lk) in CHAINS.items() for a, b, typ in lk})
NODES = {d: sorted({n for (dd, tag, x), (nd, lk) in CHAINS.items() if dd == d for n in nd}) for d in "PDG"}
print("段:", len(STEPS), " 点:", {d: len(v) for d, v in NODES.items()},
      " 着きの組:", sum(len(NODES[d]) * len(CLASSES[d]) * (2 if d == "D" else 1) for d in "PDG"))
from transformers import AutoModelForSequenceClassification
ntok = AutoTokenizer.from_pretrained(NLI_ID)
nm = AutoModelForSequenceClassification.from_pretrained(NLI_ID, torch_dtype=torch.float16).cuda().eval()
lab = {v.lower(): int(k) for k, v in nm.config.id2label.items()}
ORDER = [lab["entailment"], lab["contradiction"]]
def nli_probs(P, H, B=128):
    out = np.zeros((len(P), 2), np.float32)
    for i in range(0, len(P), B):
        enc = ntok(P[i:i + B], H[i:i + B], return_tensors="pt", padding=True, truncation=True, max_length=192).to("cuda")
        with torch.no_grad():
            pr = torch.softmax(nm(**enc).logits.float(), -1)[:, ORDER]
        out[i:i + len(pr)] = pr.cpu().numpy()
    return out
S = load_gz("E10_steps")
if S is None:
    t = time.time()
    r = nli_probs([premise(w, L[w][k]["def"]) for d, w, k, g, typ in STEPS], [step_hyp(w, g, typ, d) for d, w, k, g, typ in STEPS])
    S = [[d, w, k, g, typ, round(float(p[0]), 4), round(float(p[1]), 4)] for (d, w, k, g, typ), p in zip(STEPS, r)]
    save_gz(S, "E10_steps"); print(f"段 済み {time.time() - t:.0f}秒")
ARR = {}
for d in "PDG":
    nh = 2 if d == "D" else 1
    A = np.zeros((len(NODES[d]), len(CLASSES[d]), nh), np.float32)
    CH_N = 1000; t = time.time()
    for s in range(0, len(NODES[d]), CH_N):
        p = f"{OUT}/E10_arrive_{d}_{s}.npy"
        part = NODES[d][s:s + CH_N]
        if os.path.exists(p) and np.load(p).shape[0] == len(part):
            A[s:s + len(part)] = np.load(p); continue
        P, H = [], []
        for w, k in part:
            for c in CLASSES[d]:
                hs = arrive_hyps(w, c, d, META)
                hs = hs + [hs[0]] * (nh - len(hs))
                for h in hs: P.append(premise(w, L[w][k]["def"])); H.append(h)
        r = nli_probs(P, H)[:, 0].reshape(len(part), len(CLASSES[d]), nh)
        np.save(p, r); A[s:s + len(part)] = r
        print(f"{d} 着き {s + len(part)}/{len(NODES[d])}  {time.time() - t:.0f}秒")
    ARR[d] = A

## 6. 結果を小さく分けて保存（どれも10MBより小さく）

In [ ]:
used = sorted({n[0] for d in "PDG" for n in NODES[d]})
LX = {w: [{"def": x["def"], "labels": x["labels"], "genus": x["genus"], "loc": x["loc"], "locw": x.get("locw", {})} for x in L[w]] for w in used}
save_gz({"meta": {"nli": NLI_ID, "gemma": GEMMA_ID, "dump": DUMP_URL, "date": time.strftime("%Y-%m-%d"),
                  "items_md5": hashlib.md5(json.dumps(ITEMS).encode()).hexdigest()[:10]}, "L": LX}, "E10_L_used")
save_gz([[w, k, g, v] for (w, k, g), v in CHOICE_D.items()], "E10_choice_D")
for d in "PDG":
    ch = {f"{tag}|{x}": [[list(n) for n in nd], [[list(a), list(b), t] for a, b, t in lk]]
          for (dd, tag, x), (nd, lk) in CHAINS.items() if dd == d}
    save_gz({"chains": ch, "nodes": [list(n) for n in NODES[d]], "classes": CLASSES[d],
             "arrive": np.round(ARR[d], 3).tolist()}, f"E10_res_{d}")
print("\nドライブの E10_results の中身:")
for f in sorted(os.listdir(OUT)):
    if f.endswith(".gz"): print(f, round(os.path.getsize(f"{OUT}/{f}") / 1e6, 2), "MB")